<a href="https://colab.research.google.com/github/disiree31-abel/cs82a-portfolio/blob/main/Module_7_Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the training dataset
df = pd.read_csv("titanic_train.csv")

print(df.head())
print(df.shape)
print(df.isnull().sum())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
(8

In [2]:

# Select the features and target
X = df[["Pclass", "Sex", "Age", "Fare"]].copy()
y = df["Survived"]

# Encode sex as numbers
X["Sex"] = X["Sex"].map({"male": 0, "female": 1})

# Fill missing ages and fares with their medians
X["Age"] = X["Age"].fillna(X["Age"].median())
X["Fare"] = X["Fare"].fillna(X["Fare"].median())

print(X.head())
print("Missing values after preparation:")
print(X.isnull().sum())


   Pclass  Sex   Age     Fare
0       3    0  22.0   7.2500
1       1    1  38.0  71.2833
2       3    1  26.0   7.9250
3       1    1  35.0  53.1000
4       3    0  35.0   8.0500
Missing values after preparation:
Pclass    0
Sex       0
Age       0
Fare      0
dtype: int64


In [3]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training passengers:", len(X_train))
print("Testing passengers:", len(X_test))


Training passengers: 623
Testing passengers: 268


In [4]:

# Train a decision tree with limited depth
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

# Predict survival on the held-out test set
y_pred = model.predict(X_test)

# Calculate test accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy: {accuracy:.2%}")
print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Did not survive", "Survived"]
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


Test accuracy: 79.48%

Classification report:
                 precision    recall  f1-score   support

Did not survive       0.78      0.94      0.85       165
       Survived       0.85      0.56      0.68       103

       accuracy                           0.79       268
      macro avg       0.81      0.75      0.76       268
   weighted avg       0.80      0.79      0.78       268

Confusion matrix:
[[155  10]
 [ 45  58]]


In [5]:

# Compare predictions with actual outcomes
results = X_test.copy()
results["Actual survival"] = y_test
results["Predicted survival"] = y_pred

incorrect = results[
    results["Actual survival"] != results["Predicted survival"]
]

print("Incorrect predictions:", len(incorrect))
print(incorrect.head(10))


Incorrect predictions: 55
     Pclass  Sex   Age     Fare  Actual survival  Predicted survival
804       3    0  27.0   6.9750                1                   0
338       3    0  45.0   8.0500                1                   0
431       3    1  28.0  16.1000                1                   0
549       2    0   8.0  36.7500                1                   0
430       1    0  28.0  26.5500                1                   0
216       3    1  27.0   7.9250                1                   0
17        2    0  28.0  13.0000                1                   0
205       3    1   2.0  10.4625                0                   1
489       3    0   9.0  15.9000                1                   0
347       3    1  28.0  16.1000                1                   0


In this lab, I used a decision tree classifier to predict whether Titanic passengers survived based on passenger class, sex, age, and fare. I encoded sex as a numeric value and filled in missing ages using the median so the model could work with the data. I split the dataset into training and testing sets to evaluate how well the model predicted outcomes for passengers it had not seen before. The model incorrectly classified 55 passengers in the test set. Based on the examples, it often predicted that male passengers who survived did not survive, and it also incorrectly predicted that some female passengers who did not survive would survive. This shows that although the model can identify patterns in passenger data, it cannot accurately predict every person's outcome.